In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Load your text file
with open("source.txt", "r", encoding="utf-8") as f:
    text_data = f.read()

# 2. Configure your text splitter
# You want a chunk_size around 500, and chunk_overlap around 50
splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50) 

# 3. Create the chunks
chunks = splitter.create_documents([text_data])


# 4. Print the first 3 chunks to verify
for i in range(3):
    print(f"--- Chunk {i+1} ---")
    print(chunks[i].page_content)
    print("\n")

--- Chunk 1 ---
Quantum computing
A quantum computer is a (real or theoretical) computer that
exploits superposed and entangled states. Quantum computers
can be viewed as sampling from quantum systems. These
systems evolve in ways that operate on an enormous number
of possibilities simultaneously, though they remain subject to
strict computational constraints. By contrast, ordinary
("classical") computers operate according to deterministic
rules. (A classical computer can, in principle, be replicated by


--- Chunk 2 ---
a classical mechanical device, with only a simple multiple of
time cost. On the other hand (it is believed), a quantum
computer would require exponentially more time and energy
to be simulated classically.) It is widely believed that a
quantum computer could perform some calculations
exponentially faster than any classical computer. For
example, a large-scale quantum computer could break some
widely used public-key cryptographic schemes and aid


--- Chunk 3 ---
physic

## Embedding and storing in ChromaDB

In [2]:
import chromadb
from chromadb.utils import embedding_functions

# 5. Initialize ChromaDB (this creates a local folder to store the database)
chroma_client = chromadb.PersistentClient(path="./rag_database")

# 6. Set up the embedding model (downloads automatically on first run)
embedding_model = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

# 7. Create a collection (think of this as a table in SQL)
collection = chroma_client.get_or_create_collection(
    name="knowledge_base",
    embedding_function=embedding_model
)

# 8. Extract the text strings and create unique IDs for each chunk
documents = [chunk.page_content for chunk in chunks]
ids = [f"chunk_{i}" for i in range(len(chunks))]

# 9. Insert into the database
collection.add(
    documents=documents,
    ids=ids
)

print(f"Successfully embedded and stored {collection.count()} chunks in ChromaDB!")

c:\Users\phuc2\AppData\Local\Programs\Python\Python39\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\phuc2\AppData\Local\Programs\Python\Python39\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\phuc2\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run 

Successfully embedded and stored 266 chunks in ChromaDB!


## Building the retriever


In [3]:
# 10. Write the retrieval function
def retrieve_documents(query, k=4):
    """
    Takes a user query, embeds it, and returns the top k matching chunks from ChromaDB.
    """
    results = collection.query(
        query_texts=[query], # Chroma automatically embeds this text for you
        n_results=k
    )
    
    # Chroma returns a dictionary with lists of ids, distances, and documents.
    # We just want the raw text documents for the top match.
    return results['documents'][0]

# 11. Test the retriever
test_query = "What is a qubit and how is it different from a classical bit?"
retrieved_chunks = retrieve_documents(test_query)

# 12. Display the results
print(f"\n--- Top 4 Results for: '{test_query}' ---")
for i, chunk in enumerate(retrieved_chunks):
    print(f"\nResult {i+1}:")
    print(chunk)


--- Top 4 Results for: 'What is a qubit and how is it different from a classical bit?' ---

Result 1:
Quantum information
Just as the bit is the basic concept of classical information theory, the qubit is the fundamental unit of
quantum information. The same term qubit is used to refer to an abstract mathematical model and to any
physical system that is represented by that model. A classical bit, by definition, exists in either of two
physical states, which can be denoted 0 and 1. A qubit is also described by a state, and two states, often
written
and

Result 2:
The basic unit of information in quantum computing is the
qubit (or "quantum bit"), which serves the same function as
the bit in ordinary, or "classical", computing.[1] However,
unlike a classical bit, which can be in one of two binary
Bloch sphere representation of a qubit.
states, a qubit can exist in a linear combination of two states,
The state
is a point on
known as a quantum superposition. When one measures a
the surface

## Step 2: The Guardrail Agent (Context Evaluator)

### build an LLM function that acts as a strict filter. It will look at the user's query, look at one chunk of text, and confidently state true (keep it) or false (discard it).

In [ ]:
import os
import json
from dotenv import load_dotenv
from google import genai
from google.genai import types

# 1. Load the variables from .env
load_dotenv()

# 2. Initialize the Gemini client
llm_client = genai.Client()

# 3. Define the Guardrail Agent for Gemini
def evaluate_chunk_relevance(query, chunk_text):
    """
    Uses Gemini to grade if a chunk is relevant to the query.
    Returns a boolean: True or False.
    """
    prompt = f"""You are a strict context evaluator grading document retrieval.
    Your job is to determine if the provided document chunk contains information relevant to answering the user's query.
    
    User Query: {query}
    Document Chunk: {chunk_text}
    
    Return a JSON object with a single boolean key "relevant". 
    Set it to true if the chunk contains useful information to answer the query, otherwise false.
    """
    
    # Call Gemini and force it to return strict JSON
    response = llm_client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            temperature=0.0 # Keep it strict and deterministic
        )
    )
    
    # Parse Gemini's text response into a Python dictionary
    result = json.loads(response.text)
    return result["relevant"]

# 4. Run the retrieved chunks through the Guardrail
print("\n--- Guardrail Agent Evaluation ---")
filtered_context = []

for i, chunk in enumerate(retrieved_chunks):
    is_relevant = evaluate_chunk_relevance(test_query, chunk)
    print(f"Chunk {i+1} Relevant? {is_relevant}")
    
    if is_relevant:
        filtered_context.append(chunk)

print(f"\n{len(filtered_context)} out of {len(retrieved_chunks)} chunks survived the filter.")

c:\Users\phuc2\AppData\Local\Programs\Python\Python39\lib\site-packages\google\auth\__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
c:\Users\phuc2\AppData\Local\Programs\Python\Python39\lib\site-packages\google\oauth2\__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)



--- Guardrail Agent Evaluation ---
Chunk 1 Relevant? True
Chunk 2 Relevant? True
Chunk 3 Relevant? True
Chunk 4 Relevant? False

3 out of 4 chunks survived the filter.
